In [1]:
import pandas as pd
import os
import glob
from pathlib import Path
import numpy as np
folder = Path(r"D:\studystudy\Window Dressing\raw data\stk price")
files = sorted(folder.glob("TRD_BwardQuotationMonth*.csv"))
df_all_price = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
df_all.drop(columns = ['Filling','CirculatedMarketValue'],inplace = True)

In [2]:
df2 = df_all.copy()

# 1) dtype 一次性处理（尽量用矢量化、避免反复 astype/assign）
df2["CloseDate"] = pd.to_datetime(df2["CloseDate"], errors="coerce")

# TradingMonth：建议统一成 YYYYMM 的整数，比较快也省内存
# 如果原来是 "2024-01" / "202401" / 202401 混杂，这段会尽量规整
tm = df2["TradingMonth"].astype(str).str.replace("-", "", regex=False).str.slice(0, 6)
df2["TradingMonth_int"] = pd.to_numeric(tm, errors="coerce").astype("Int64")

df2["Year"] = df2["CloseDate"].dt.year
df2["Half"] = (df2["CloseDate"].dt.month > 6).astype("int8") + 1  # 1/2

# 2) 一次排序，后面 idx/first/last 都能用
df2 = df2.sort_values(["Symbol", "Year", "Half", "CloseDate"], kind="mergesort")

gkey = ["Symbol", "Year", "Half"]

# 3) 期末：每半年最后交易日
end_idx = df2.groupby(gkey, sort=False)["CloseDate"].idxmax()
end_px = (
    df2.loc[end_idx, ["Symbol", "Year", "Half", "CloseDate", "ClosePrice"]]
       .rename(columns={"CloseDate": "EndDate", "ClosePrice": "EndClose"})
)

# 4) 期初：每半年内最早 TradingMonth 的“该月第一条记录”的 OpenPrice
# 先找每个半年组最小 TradingMonth
min_tm = df2.groupby(gkey, sort=False)["TradingMonth_int"].min()

# 给每行标记：是否属于该组的最小 TradingMonth
df2 = df2.join(min_tm.rename("min_tm"), on=gkey)
mask = df2["TradingMonth_int"].eq(df2["min_tm"])

# 在最小 TradingMonth 子集里，取每组第一条（因为已按 CloseDate 排序）
start_px = (
    df2[mask]
      .groupby(gkey, sort=False)
      .first()[["TradingMonth_int", "OpenPrice"]]
      .reset_index()
      .rename(columns={"TradingMonth_int": "StartMonth", "OpenPrice": "StartOpen"})
)

# 5) 合并 + 计算收益率/得分
half_ret = (
    start_px.merge(end_px, on=gkey, how="inner")
            .assign(
                EndMonth=lambda x: x["EndDate"].dt.to_period("M"),
                HalfReturn=lambda x: x["EndClose"] / x["StartOpen"]
            )
            .sort_values(["Symbol", "Year", "Half"])
)

half_ret["WinnerScore"] = (
    half_ret.groupby(["Year", "Half"], sort=False)["HalfReturn"]
            .rank(pct=True, method="average")
            .mul(5).pipe(np.ceil)
            .clip(1, 5)
            .astype("int8")
)

half_ret['HalfReturn'] = half_ret['HalfReturn'] - 1


In [3]:
half_ret.to_csv(r'D:\studystudy\Window Dressing\merged data\merged_half_year_stk_ret.csv', index=False, sep=',', encoding='utf-8-sig')

In [5]:
half_ret

,Symbol,Year,Half,StartMonth,StartOpen,EndDate,EndClose,EndMonth,HalfReturn,WinnerScore
0,1,2005,1,200501,223.392,2005-06-30,201.019,2005-06,0.899849,4
1,1,2005,2,200507,200.680,2005-12-30,208.137,2005-12,1.037159,3
2,1,2006,1,200601,208.137,2006-06-30,256.273,2006-06,1.231271,1
3,1,2006,2,200607,256.273,2006-12-29,490.513,2006-12,1.914025,5
4,1,2007,1,200701,490.513,2007-06-29,1026.500,2007-06,2.092707,3
...,...,...,...,...,...,...,...,...,...,...
123738,920992,2022,2,202210,16.010,2022-12-30,11.660,2022-12,0.728295,1
123739,920992,2023,1,202301,11.660,2023-06-30,9.108,2023-06,0.781132,1
123740,920992,2023,2,202307,9.118,2023-12-29,13.026,2023-12,1.428603,5
123741,920992,2024,1,202401,13.026,2024-06-28,8.848,2024-06,0.679257,2
